# 02 - Embedding Experiments

Objective: validate the Phase 2 job embeddings used by retrieval, clustering, and salary modeling.

This notebook answers four practical questions:
- Do the generated embeddings have the expected shape, dtype, and normalization?
- Do low-dimensional projections show interpretable structure by role, location, or experience level?
- Does the FAISS index align with the embedding rows and metadata rows?
- How should we run optional model comparisons such as MiniLM vs. mpnet without making the notebook require internet access?

Success criteria for the project demo: `models/job_embeddings.npy`, `models/jobs.index`, and `models/jobs_meta.parquet` exist, have aligned row counts, and can support plausible top-k retrieval checks.

## How to use this notebook

Run from the repo root or from the `notebooks/` directory. The notebook is safe to run before data artifacts exist: missing artifacts produce setup instructions instead of hard failures.

To create the real artifacts first:

```bash
uv run python scripts/preprocess_data.py
uv run python scripts/build_index.py
```

Model comparison and hand-crafted text retrieval are opt-in because they may download Hugging Face weights on a fresh machine.

In [1]:
from __future__ import annotations

import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

EMBEDDINGS_PATH = PROJECT_ROOT / "models" / "job_embeddings.npy"
INDEX_PATH = PROJECT_ROOT / "models" / "jobs.index"
META_PATH = PROJECT_ROOT / "models" / "jobs_meta.parquet"
JOBS_PATH = PROJECT_ROOT / "data" / "processed" / "jobs.parquet"

RANDOM_SEED = 42
SAMPLE_SIZE = 800
TOP_K = 5

# Keep these False for clean offline execution. Toggle locally when the
# sentence-transformer models are cached or internet access is available.
RUN_OPTIONAL_MODEL_COMPARISON = False
RUN_HANDCRAFTED_RETRIEVAL = False

rng = np.random.default_rng(RANDOM_SEED)
pd.set_option("display.max_colwidth", 120)

## 1. Artifact availability

In [2]:
def file_size_mb(path: Path) -> float | None:
    return round(path.stat().st_size / (1024**2), 2) if path.exists() else None


artifacts = pd.DataFrame(
    [
        {
            "artifact": "job embeddings",
            "path": EMBEDDINGS_PATH.relative_to(PROJECT_ROOT),
            "exists": EMBEDDINGS_PATH.exists(),
            "size_mb": file_size_mb(EMBEDDINGS_PATH),
        },
        {
            "artifact": "FAISS index",
            "path": INDEX_PATH.relative_to(PROJECT_ROOT),
            "exists": INDEX_PATH.exists(),
            "size_mb": file_size_mb(INDEX_PATH),
        },
        {
            "artifact": "job metadata",
            "path": META_PATH.relative_to(PROJECT_ROOT),
            "exists": META_PATH.exists(),
            "size_mb": file_size_mb(META_PATH),
        },
        {
            "artifact": "processed jobs",
            "path": JOBS_PATH.relative_to(PROJECT_ROOT),
            "exists": JOBS_PATH.exists(),
            "size_mb": file_size_mb(JOBS_PATH),
        },
    ]
)
display(artifacts)

missing = artifacts.loc[~artifacts["exists"], "path"].astype(str).tolist()
if missing:
    print("Missing artifacts:")
    for path in missing:
        print(f"  - {path}")
    print()
    print("Generate them with:")
    print("  uv run python scripts/preprocess_data.py")
    print("  uv run python scripts/build_index.py")
else:
    print("All embedding experiment artifacts are present.")

         artifact                         path  exists  size_mb
0  job embeddings    models/job_embeddings.npy    True    51.44
1     FAISS index            models/jobs.index    True    51.44
2    job metadata     models/jobs_meta.parquet    True     1.37
3  processed jobs  data/processed/jobs.parquet    True   220.72
All embedding experiment artifacts are present.


## 2. Load embeddings and metadata

In [3]:
HAS_EMBEDDINGS = EMBEDDINGS_PATH.exists() and META_PATH.exists()

if HAS_EMBEDDINGS:
    embeddings = np.load(EMBEDDINGS_PATH, mmap_mode="r")
    metadata = pd.read_parquet(META_PATH)
    jobs = pd.read_parquet(JOBS_PATH) if JOBS_PATH.exists() else None

    print(f"embeddings shape: {embeddings.shape}")
    print(f"embeddings dtype:  {embeddings.dtype}")
    print(f"metadata rows:     {len(metadata):,}")

    assert embeddings.ndim == 2, "Expected a 2D embedding matrix"
    assert len(metadata) == embeddings.shape[0], "Metadata rows must align with embedding rows"
    assert embeddings.dtype == np.float32, "build_index.py should write float32 vectors"
else:
    embeddings = np.empty((0, 0), dtype=np.float32)
    metadata = pd.DataFrame()
    jobs = None
    print("Skipping load: embeddings and metadata are not available yet.")

embeddings shape: (35118, 384)
embeddings dtype:  float32
metadata rows:     35,118


## 3. Embedding quality checks

In [4]:
if HAS_EMBEDDINGS:
    probe_size = min(5000, embeddings.shape[0])
    probe_idx = rng.choice(embeddings.shape[0], size=probe_size, replace=False)
    probe = np.asarray(embeddings[probe_idx], dtype=np.float32)
    norms = np.linalg.norm(probe, axis=1)

    quality = pd.DataFrame(
        {
            "metric": [
                "rows",
                "dimension",
                "dtype_is_float32",
                "finite_values",
                "norm_min",
                "norm_mean",
                "norm_max",
            ],
            "value": [
                embeddings.shape[0],
                embeddings.shape[1],
                embeddings.dtype == np.float32,
                np.isfinite(probe).all(),
                norms.min(),
                norms.mean(),
                norms.max(),
            ],
        }
    )
    display(quality)
else:
    print("Quality checks skipped because embeddings are missing.")

             metric  value
0              rows  35118
1         dimension    384
2  dtype_is_float32   True
3     finite_values   True
4          norm_min    1.0
5         norm_mean    1.0
6          norm_max    1.0


Expected result: vectors should be `float32`, two-dimensional, finite, and row-wise L2-normalized. For the current MiniLM index, the expected shape is `(n_jobs, 384)`.

## 4. Sample embeddings for projection

In [5]:
if HAS_EMBEDDINGS:
    sample_n = min(SAMPLE_SIZE, embeddings.shape[0])
    sample_idx = rng.choice(embeddings.shape[0], size=sample_n, replace=False)
    sample_embeddings = np.asarray(embeddings[sample_idx], dtype=np.float32)
    sample_meta = metadata.iloc[sample_idx].reset_index(drop=True).copy()
    sample_meta["row_id"] = sample_idx

    display(
        sample_meta[
            [
                "row_id",
                "title",
                "company_name",
                "location",
                "experience_level",
                "salary_annual",
            ]
        ].head()
    )
    print(f"Projection sample: {sample_embeddings.shape}")
else:
    sample_embeddings = np.empty((0, 0), dtype=np.float32)
    sample_meta = pd.DataFrame()
    print("Sampling skipped because embeddings are missing.")

   row_id  ... salary_annual
0   14416  ...      126880.0
1   19307  ...      135000.0
2   21384  ...      135200.0
3   29808  ...       60000.0
4    7002  ...      135000.0

[5 rows x 6 columns]
Projection sample: (800, 384)


## 5. PCA projection

In [6]:
if HAS_EMBEDDINGS and len(sample_embeddings) >= 3:
    n_components = min(10, sample_embeddings.shape[1], len(sample_embeddings))
    pca = PCA(n_components=n_components, random_state=RANDOM_SEED)
    pca_coords = pca.fit_transform(sample_embeddings)

    variance = pd.DataFrame(
        {
            "component": [f"PC{i + 1}" for i in range(n_components)],
            "explained_variance_ratio": pca.explained_variance_ratio_,
            "cumulative": np.cumsum(pca.explained_variance_ratio_),
        }
    )
    display(variance)

    plot_df = sample_meta.copy()
    plot_df["pc1"] = pca_coords[:, 0]
    plot_df["pc2"] = pca_coords[:, 1] if n_components > 1 else 0.0
    plot_df["salary_annual"] = pd.to_numeric(plot_df["salary_annual"], errors="coerce")

    fig = px.scatter(
        plot_df,
        x="pc1",
        y="pc2",
        color="experience_level",
        hover_data=["title", "company_name", "location", "salary_annual"],
        title="PCA view of sampled job embeddings",
        height=620,
    )
    fig.show()
else:
    print("PCA skipped because embeddings are missing or the sample is too small.")

  component  explained_variance_ratio  cumulative
0       PC1                  0.063283    0.063283
1       PC2                  0.041619    0.104901
2       PC3                  0.035618    0.140519
3       PC4                  0.035092    0.175611
4       PC5                  0.031904    0.207515
5       PC6                  0.023363    0.230879
6       PC7                  0.020746    0.251624
7       PC8                  0.018634    0.270258
8       PC9                  0.017425    0.287683
9      PC10                  0.015284    0.302967
[plot rendered: PCA view of sampled job embeddings]


## 6. t-SNE projection

In [7]:
if HAS_EMBEDDINGS and len(sample_embeddings) >= 50:
    perplexity = min(30, max(5, (len(sample_embeddings) - 1) // 3))
    start = time.perf_counter()
    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        metric="cosine",
        init="pca",
        learning_rate="auto",
        random_state=RANDOM_SEED,
    )
    tsne_coords = tsne.fit_transform(sample_embeddings)
    elapsed = time.perf_counter() - start
    print(
        f"t-SNE completed on {len(sample_embeddings):,} vectors "
        f"in {elapsed:.1f}s with perplexity={perplexity}."
    )

    tsne_df = sample_meta.copy()
    tsne_df["tsne_1"] = tsne_coords[:, 0]
    tsne_df["tsne_2"] = tsne_coords[:, 1]

    fig = px.scatter(
        tsne_df,
        x="tsne_1",
        y="tsne_2",
        color="experience_level",
        hover_data=["title", "company_name", "location", "salary_annual"],
        title="t-SNE view of sampled job embeddings",
        height=620,
    )
    fig.show()
else:
    print("t-SNE skipped because embeddings are missing or fewer than 50 rows are available.")

t-SNE completed on 800 vectors in 1.5s with perplexity=30.
[plot rendered: t-SNE view of sampled job embeddings]


## 7. Optional model comparison: MiniLM vs. mpnet

The default production model is `all-MiniLM-L6-v2` because it is fast and produces 384-dimensional vectors. A larger alternative such as `all-mpnet-base-v2` usually produces 768-dimensional vectors and may improve semantic quality at higher compute cost.

This cell is opt-in. Set `RUN_OPTIONAL_MODEL_COMPARISON = True` in the setup cell only when model weights are cached locally or internet access is available.

In [8]:
COMPARISON_MODELS = ["all-MiniLM-L6-v2", "all-mpnet-base-v2"]
COMPARISON_TEXTS = [
    "Senior machine learning engineer building retrieval systems with Python, FAISS, and PyTorch.",
    "Data analyst role focused on SQL dashboards, experimentation, and stakeholder reporting.",
    "Frontend engineer building Streamlit and React product prototypes for job search workflows.",
    "Quantitative salary modeling with regression, calibration, and uncertainty estimates.",
]

if RUN_OPTIONAL_MODEL_COMPARISON:
    from ml.embeddings import Encoder

    rows = []
    encoded_by_model = {}
    for model_name in COMPARISON_MODELS:
        start = time.perf_counter()
        encoder = Encoder(model_name=model_name)
        vectors = encoder.encode(COMPARISON_TEXTS)
        elapsed = time.perf_counter() - start
        encoded_by_model[model_name] = vectors
        rows.append(
            {
                "model": model_name,
                "shape": vectors.shape,
                "dtype": vectors.dtype,
                "mean_norm": np.linalg.norm(vectors, axis=1).mean(),
                "seconds": round(elapsed, 2),
            }
        )
    display(pd.DataFrame(rows))

    for model_name, vectors in encoded_by_model.items():
        cosine = vectors @ vectors.T
        print()
        print(f"Pairwise cosine similarities: {model_name}")
        display(
            pd.DataFrame(
                cosine,
                index=range(len(COMPARISON_TEXTS)),
                columns=range(len(COMPARISON_TEXTS)),
            ).round(3)
        )
else:
    display(
        pd.DataFrame(
            [
                {
                    "model": "all-MiniLM-L6-v2",
                    "expected_dim": 384,
                    "notes": "current production default; fast enough for local indexing",
                },
                {
                    "model": "all-mpnet-base-v2",
                    "expected_dim": 768,
                    "notes": "optional comparison; slower and larger, may improve retrieval quality",
                },
            ]
        )
    )
    print("Skipped live model comparison. Toggle RUN_OPTIONAL_MODEL_COMPARISON=True to run it locally.")

               model  ...                                                                  notes
0   all-MiniLM-L6-v2  ...             current production default; fast enough for local indexing
1  all-mpnet-base-v2  ...  optional comparison; slower and larger, may improve retrieval quality

[2 rows x 3 columns]
Skipped live model comparison. Toggle RUN_OPTIONAL_MODEL_COMPARISON=True to run it locally.


## 8. FAISS index alignment check

In [9]:
if HAS_EMBEDDINGS and INDEX_PATH.exists():
    import faiss

    index = faiss.read_index(str(INDEX_PATH))
    print(f"FAISS rows: {index.ntotal:,}; dimension: {index.d}")
    assert index.ntotal == embeddings.shape[0], "FAISS index row count must match embeddings"
    assert index.d == embeddings.shape[1], "FAISS dimension must match embeddings"

    query_rows = sample_idx[: min(5, len(sample_idx))]
    query_vectors = np.asarray(embeddings[query_rows], dtype=np.float32)
    scores, neighbors = index.search(query_vectors, TOP_K)

    rows = []
    for q_pos, source_row in enumerate(query_rows):
        source = metadata.iloc[source_row]
        for rank, neighbor_row in enumerate(neighbors[q_pos], start=1):
            neighbor = metadata.iloc[int(neighbor_row)]
            rows.append(
                {
                    "query_row": int(source_row),
                    "query_title": source["title"],
                    "rank": rank,
                    "neighbor_row": int(neighbor_row),
                    "score": float(scores[q_pos, rank - 1]),
                    "neighbor_title": neighbor["title"],
                    "neighbor_company": neighbor["company_name"],
                }
            )
    display(pd.DataFrame(rows))

    self_match_rate = float(np.mean(neighbors[:, 0] == query_rows))
    print(f"Top-1 self-match rate for sampled stored vectors: {self_match_rate:.2%}")
else:
    print("FAISS alignment check skipped because embeddings/index artifacts are missing.")

FAISS rows: 35,118; dimension: 384
    query_row  ...                         neighbor_company
0       14416  ...  ML OUTSOURCING SERVICES PRIVATE LIMITED
1       14416  ...                              ACL Digital
2       14416  ...                            Cube Hub Inc.
3       14416  ...                          Net2Source Inc.
4       14416  ...                             SPECTRAFORCE
5       19307  ...                                    AECOM
6       19307  ...                                    AECOM
7       19307  ...                                    AECOM
8       19307  ...                                    AECOM
9       19307  ...                                    AECOM
10      21384  ...                     Marc Fisher Footwear
11      21384  ...                    STEM Talent Solutions
12      21384  ...        Creative Financial Staffing (CFS)
13      21384  ...                                  Swooped
14      21384  ...                              Robert Half
15   

The self-query check should return each stored vector as its own top-1 neighbor. If this fails, the index, embeddings, and metadata are not row-aligned.

## 9. Optional hand-crafted retrieval checks

In [10]:
HANDCRAFTED_QUERIES = {
    "ml_engineer": "Machine learning engineer with Python, PyTorch, embeddings, recommender systems, and FAISS retrieval experience.",
    "data_analyst": "Data analyst with SQL, dashboards, experimentation, business intelligence, and stakeholder reporting experience.",
    "product_manager": "Product manager for search and recommendation products, roadmap planning, user research, and cross-functional delivery.",
}

if RUN_HANDCRAFTED_RETRIEVAL and HAS_EMBEDDINGS and INDEX_PATH.exists():
    import faiss
    from ml.embeddings import Encoder

    index = faiss.read_index(str(INDEX_PATH))
    encoder = Encoder()
    rows = []
    for label, query in HANDCRAFTED_QUERIES.items():
        query_vec = encoder.encode([query])
        scores, neighbors = index.search(query_vec, TOP_K)
        for rank, neighbor_row in enumerate(neighbors[0], start=1):
            match = metadata.iloc[int(neighbor_row)]
            rows.append(
                {
                    "query": label,
                    "rank": rank,
                    "score": float(scores[0, rank - 1]),
                    "title": match["title"],
                    "company": match["company_name"],
                    "location": match["location"],
                    "experience_level": match["experience_level"],
                    "salary_annual": match["salary_annual"],
                }
            )
    display(pd.DataFrame(rows))
else:
    print("Skipped hand-crafted retrieval to avoid model downloads during default notebook execution.")
    print("Set RUN_HANDCRAFTED_RETRIEVAL=True after the default SentenceTransformer model is cached locally.")

Skipped hand-crafted retrieval to avoid model downloads during default notebook execution.
Set RUN_HANDCRAFTED_RETRIEVAL=True after the default SentenceTransformer model is cached locally.


## 10. Notes for clustering

Clustering should consume `models/job_embeddings.npy` directly. The current production artifact is expected to be a float32 matrix with one row per processed job and one column per embedding dimension.

In [11]:
if HAS_EMBEDDINGS:
    clustering_notes = pd.DataFrame(
        [
            {"property": "shape", "value": str(tuple(embeddings.shape))},
            {"property": "dtype", "value": str(embeddings.dtype)},
            {"property": "row alignment", "value": "row i matches jobs_meta.parquet row i"},
            {"property": "distance assumption", "value": "L2-normalized; inner product equals cosine similarity"},
        ]
    )
    display(clustering_notes)
else:
    print("Clustering notes skipped because embeddings are missing.")

              property                                                  value
0                shape                                           (35118, 384)
1                dtype                                                float32
2        row alignment                  row i matches jobs_meta.parquet row i
3  distance assumption  L2-normalized; inner product equals cosine similarity


## Conclusions and follow-ups

Real-data run completed on the generated Phase 2 artifacts.

Key results:
- `models/job_embeddings.npy` loaded successfully with shape `(35118, 384)` and dtype `float32`.
- `models/jobs_meta.parquet` loaded with `35,118` rows, matching the embedding row count.
- Sampled embedding norms were stable: min `1.0`, mean `1.0`, max `1.0`, confirming row-wise L2 normalization.
- PCA on an 800-job sample showed diffuse semantic structure rather than one dominant axis: PC1 explained `6.3%`, PC2 explained `4.2%`, and the first 10 PCs explained about `30.3%` cumulatively.
- t-SNE ran successfully on the 800-job sample with cosine distance and perplexity `30`.
- FAISS index alignment passed: `models/jobs.index` has `35,118` vectors, dimension `384`, and stored-vector self-query top-1 match rate was `100.00%`.
- Optional live MiniLM-vs-mpnet and hand-crafted retrieval cells were intentionally left off by default so the notebook stays offline-safe and does not download model weights unexpectedly.

Project implications:
- The embedding artifact is ready for clustering, salary modeling, and app integration.
- For retrieval quality tuning, the next useful experiment is to enable hand-crafted retrieval checks and compare `all-MiniLM-L6-v2` against `all-mpnet-base-v2` if model downloads are available.
